In [ ]:
%pip install langchain langchain-openai langchain-community langgraph langgraph-prebuilt wikipedia --upgrade --quiet

Tools are interfaces that an agent, chain, or LLM can use to interact with the world. They combine a few things:

- The name of the tool
- A description of what the tool is
- JSON schema of what the inputs to the tool are
- The function to call
- Whether the result of a tool should be returned directly to the user

It is useful to have all this information because this information can be used to build action-taking systems! The name, description, and JSON schema can be used to prompt the LLM so it knows how to specify what action to take, and then the function to call is equivalent to taking that action.

In [ ]:
import getpass
import os

# Set up your OpenAI client
if not os.getenv("OPENAI_API_KEY"):
    secret_key = getpass.getpass("Enter your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = secret_key

In [ ]:
# 1. Standard Tools
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

api_wrapper = WikipediaAPIWrapper(top_k_results=1, doc_content_chars_max=1000)
tool = WikipediaQueryRun(api_wrapper=api_wrapper)

print(tool.name)
print(tool.description)
print(tool.args)

# We can see if the tool should return directly to the user
print("Will this automatically return the output to the user? This value is a boolean:", tool.return_direct)

## How to Call The Tool Without An Agent:

In [ ]:
tool.invoke("What is Digital Marketing?")

# 2. Creating Custom Tools
When constructing your own agent, you will need to provide it with a list of Tools that it can use. Besides the actual function that is called, the Tool consists of several components:

- `name (str)`, is required and must be unique within a set of tools provided to an agent
- `description (str)`, is optional but recommended, as it is used by an agent to determine tool use
- `args_schema (Pydantic BaseModel)`, is optional but recommended, can be used to provide more information (e.g., few-shot examples) or validation for expected parameters.

In [ ]:
# Import things that are needed generically
from pydantic import BaseModel, Field
from langchain_core.tools import BaseTool, StructuredTool, tool

## `@tool decorator`

This `@tool` decorator is the simplest way to define a custom tool. The decorator uses the function name as the tool name by default, but this can be overridden by passing a string as the first argument. Additionally, the decorator will use the function’s docstring as the tool’s description - so a docstring MUST be provided.

In [ ]:
@tool
def search(query: str) -> str:
    """Look up things online."""
    return "LangChain"

print(search.name)
print(search.description)
print(search.args)

You can also customize the tool name and JSON args by passing them into the tool decorator.

In [ ]:
class SearchInput(BaseModel):
    query: str = Field(description="should be a search query")

@tool("search-tool", args_schema=SearchInput, return_direct=True)
def search(query: str) -> str:
    """Look up things online."""
    return "LangChain"

## StructuredTool dataclass

In [ ]:
def search_function(query: str):
    return "LangChain"

search = StructuredTool.from_function(
    func=search_function,
    name="Search",
    description="useful for when you need to answer questions about current events",
    # coroutine= ... <- you can specify an async method if desired as well
)

--------

## [Agents](https://python.langchain.com/docs/concepts/agents/)

The core idea of agents is to use a language model to choose a sequence of actions to take. In chains, a sequence of actions is hardcoded (in code). In agents, a language model is used as a reasoning engine to determine which actions to take and in which order.

LangChain provides `create_agent` which builds a tool-calling agent that automatically handles the tool execution loop.

In [ ]:
from langchain_core.tools import tool

# 1. Create the tool:
@tool
def get_word_length(word: str) -> int:
    """Returns the length of a word."""
    return len(word)

# 2. Assign the tools to a Python list:
tools = [get_word_length]

## Creating an Agent with `create_agent`

The `create_agent` function from `langchain.agents` is the modern way to build agents. It takes a model, tools, and an optional system prompt. It automatically handles tool binding, the tool execution loop, and message formatting.

## How does the agent know what tools it can use?

We're relying on OpenAI tool calling LLMs, which take tools as a separate argument and have been specifically trained to know when to invoke those tools. `create_agent` handles binding the tools to the LLM automatically.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent

llm = ChatOpenAI()

# create_agent handles tool binding and the tool execution loop automatically
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a very powerful assistant, but don't know current events.",
)

## Invoking the Agent

With `create_agent`, we pass messages in and get messages back. The agent automatically decides when to call tools and returns the final response.

In [ ]:
from langchain_core.messages import HumanMessage

result = agent.invoke({"messages": [HumanMessage("How many letters in the word data?")]})

for m in result["messages"]:
    m.pretty_print()

In [ ]:
# You can also stream events from the agent:
for event in agent.stream({"messages": [HumanMessage("How many letters in the word langchain?")]}):
    for key, value in event.items():
        print(f"--- {key} ---")
        for m in value.get("messages", []):
            m.pretty_print()

In [ ]:
# Access just the final output:
print(result["messages"][-1].content)

---

## Adding Memory with a Checkpointer

The modern way to add memory to an agent is with a **checkpointer**. The `MemorySaver` checkpointer stores conversation state in memory, keyed by a `thread_id`. This means:

- Same `thread_id` = same conversation (the agent remembers previous messages)
- Different `thread_id` = fresh conversation (no memory of previous messages)

This replaces the old `ConversationBufferMemory` and `RunnableWithMessageHistory` patterns.

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# Create a new agent with a checkpointer for memory:
agent_with_memory = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a very powerful assistant, but bad at calculating lengths of words.",
    checkpointer=MemorySaver(),
)

In [ ]:
# Use a thread_id to maintain conversation state:
config = {"configurable": {"thread_id": "session-1"}}

result = agent_with_memory.invoke(
    {"messages": [HumanMessage("How many letters in the word data?")]},
    config=config,
)
print(result["messages"][-1].content)

In [ ]:
# Follow-up question using the same thread_id — the agent remembers the previous exchange:
result = agent_with_memory.invoke(
    {"messages": [HumanMessage("Is that a real word?")]},
    config=config,
)
print(result["messages"][-1].content)

---

## Isolating Conversations with Different `thread_id`s

Each `thread_id` creates an isolated conversation. This is how you'd handle multiple users or sessions.

In [ ]:
# First session — introduce ourselves:
config_1 = {"configurable": {"thread_id": "user-james"}}

result = agent_with_memory.invoke(
    {"messages": [HumanMessage("My name is James")]},
    config=config_1,
)
print("Session 1:", result["messages"][-1].content)

In [ ]:
# Same session — the agent remembers our name:
result = agent_with_memory.invoke(
    {"messages": [HumanMessage("What is my name?")]},
    config=config_1,
)
print("Session 1:", result["messages"][-1].content)

In [ ]:
# Different session — the agent does NOT know our name:
config_2 = {"configurable": {"thread_id": "user-unknown"}}

result = agent_with_memory.invoke(
    {"messages": [HumanMessage("What is my name?")]},
    config=config_2,
)
print("Session 2:", result["messages"][-1].content)

## Summary

- **Tools** are interfaces that agents use to interact with the world. You can use built-in tools (e.g. `WikipediaQueryRun`) or create custom ones with `@tool`, `StructuredTool`, or `BaseTool`.
- **`create_agent`** from `langchain.agents` is the modern way to build agents. It handles tool binding and the tool execution loop automatically.
- **Memory** is added via a **checkpointer** (e.g. `MemorySaver`). Each conversation is isolated by a unique `thread_id`.